In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


BASE_DIR     = "/kaggle/input/datasets/xza3tar/dataset-for-h-d-k"
KIDNEY_PATH  = "/kaggle/input/datasets/mansoordaku/ckdisease/kidney_disease.csv"


# -------------------------------------------------------------------
# Dataset configs
# -------------------------------------------------------------------

diabetes_config = {
    "path": f"{BASE_DIR}/diabetes_prediction_dataset.csv",
    "target": "diabetes",
    "features": [
        "HbA1c_level", "blood_glucose_level", "age", "bmi",
        "smoking_history", "hypertension", "gender", "heart_disease"
    ]
}

heart_config = {
    "path": f"{BASE_DIR}/heart_2022_no_nans.csv",
    "target": "HadHeartAttack",
    "features": [
        "HadAngina", "ChestScan", "HadStroke", "DifficultyWalking",
        "HadDiabetes", "GeneralHealth", "HadArthritis", "PneumoVaxEver",
        "RemovedTeeth", "AgeCategory", "SmokerStatus", "BMI",
        "HadKidneyDisease", "HadCOPD"
    ]
}

kidney_config = {
    "path": KIDNEY_PATH,
    "target": "classification",
    "features": [
        "age", "bp", "sg", "al", "su",
        "rbc", "pc", "pcc", "ba",
        "bgr", "bu", "sc", "sod", "pot",
        "hemo", "pcv", "wc", "rc",
        "htn", "dm", "cad", "appet", "pe", "ane"
    ]
}

DATASETS = {
    "diabetes": diabetes_config,
    "heart":    heart_config,
    "kidney":   kidney_config
}


# -------------------------------------------------------------------
# Models
# -------------------------------------------------------------------

def make_xgb():
    return CalibratedClassifierCV(
        XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            min_child_weight=10,
            subsample=0.8,
            colsample_bytree=0.7,
            scale_pos_weight=1,
            use_label_encoder=False,
            eval_metric="logloss",
            tree_method="hist",
            device="cuda",
            random_state=42
        ),
        method="isotonic",
        cv=3
    )


# -------------------------------------------------------------------
# Data cleaning
# -------------------------------------------------------------------

def fix_value(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    return np.nan if x in ["?", "", "nan", "none"] else x

def to_binary(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float, np.integer, np.floating)):
        return 1 if float(v) != 0 else 0
    s = str(v).strip().lower()
    if s in {"0", "no", "negative", "normal", "false", "notckd"}: return 0
    if s in {"1", "yes", "positive", "abnormal", "true", "ckd"}:  return 1
    return 0 if "not" in s else 1

def prepare_diabetes(df):
    df = df.copy()
    df["smoking_history"] = df["smoking_history"].replace("No Info", np.nan)
    return df

def prepare_kidney(df):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]

    # fix dirty string values in all object cols
    for col in df.select_dtypes("object").columns:
        df[col] = df[col].apply(fix_value)

    # fix target — ckd\t should be ckd
    df["classification"] = df["classification"].replace("ckd\t", "ckd")

    # pcv, wc, rc are numeric but stored as strings
    for col in ["pcv", "wc", "rc"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # drop id — not a feature
    if "id" in df.columns:
        df = df.drop(columns=["id"])

    return df

def prepare_data(df, disease):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    if disease == "diabetes": return prepare_diabetes(df)
    if disease == "kidney":   return prepare_kidney(df)
    return df


# -------------------------------------------------------------------
# Preprocessor
# -------------------------------------------------------------------

def build_preprocessor(X):
    cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler())
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ])

    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])


# -------------------------------------------------------------------
# Training
# -------------------------------------------------------------------

def train_model(df, feature_cols, target_col, disease):
    df = prepare_data(df, disease)
    df = df[feature_cols + [target_col]].copy()

    X = df[feature_cols].copy()
    y = df[target_col].apply(to_binary)

    X = X[y.notna()]
    y = y[y.notna()].astype(int)

    print(f"\nClass distribution:\n{y.value_counts()}\n")

    preprocessor = build_preprocessor(X)

    # kidney dataset is already balanced — no SMOTE needed
    if disease == "kidney":
        model = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier",   make_xgb())
        ])
    else:
        model = ImbPipeline([
            ("preprocessor", preprocessor),
            ("smote",        SMOTE(sampling_strategy=0.6, random_state=42)),
            ("classifier",   make_xgb())
        ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print(f"\n{disease.upper()} — Test Results")
    print(classification_report(y_test, y_pred))

    return model, X


# -------------------------------------------------------------------
# User input
# -------------------------------------------------------------------

def ask_value(col, series):
    if pd.api.types.is_numeric_dtype(series):
        while True:
            val = input(f"  {col}: ").strip()
            try:
                return float(val)
            except ValueError:
                print("  Please enter a valid number.")
    else:
        options = sorted(series.dropna().astype(str).unique().tolist())
        print(f"\n  Options for '{col}': {options[:15]}")
        val = input(f"  {col}: ").strip()

        for option in options:
            if option.lower() == val.lower():
                return option

        return val

def collect_patient_data(reference_df, feature_cols):
    print("\nEnter the patient's information:\n")
    data = {col: ask_value(col, reference_df[col]) for col in feature_cols}
    return pd.DataFrame([data])


# -------------------------------------------------------------------
# Show result
# -------------------------------------------------------------------

def show_result(pred, prob, disease):
    print("\n" + "=" * 50)
    print("RESULT")
    print("=" * 50)

    print("Likely DISEASED ⚠" if pred == 1 else "Likely HEALTHY ✓")

    if prob < 0.20:
        risk, desc = "🟢 Low Risk",       "No significant indicators detected."
    elif prob < 0.50:
        risk, desc = "🟡 Moderate Risk",  "Some risk factors present. Consider seeing a doctor."
    elif prob < 0.80:
        risk, desc = "🟠 High Risk",      "Multiple risk factors found. See a doctor."
    else:
        risk, desc = "🔴 Very High Risk", "Strong disease indicators. Please see a doctor immediately."

    print(f"Risk   : {risk}")
    print(f"       {desc}")
    print("=" * 50)


# -------------------------------------------------------------------
# Main
# -------------------------------------------------------------------

print("=" * 50)
print("Disease Prediction System")
print("=" * 50)
print("1 - Diabetes")
print("2 - Heart Disease")
print("3 - Kidney Disease")
print("=" * 50)

choice = input("Choose (1/2/3): ").strip()

disease_map = {"1": "diabetes", "2": "heart", "3": "kidney"}

if choice not in disease_map:
    raise ValueError("Invalid choice. Please enter 1, 2, or 3.")

disease = disease_map[choice]
config  = DATASETS[disease]

print(f"\nLoading {disease} data and training model...")
df       = pd.read_csv(config["path"])
model, X = train_model(df, config["features"], config["target"], disease)
print(f"\n✓ {disease.capitalize()} model is ready.")

user_df = collect_patient_data(X, config["features"])

if disease == "diabetes":
    user_df["smoking_history"] = user_df["smoking_history"].replace("No Info", np.nan)

if disease == "kidney":
    for col in user_df.select_dtypes("object").columns:
        user_df[col] = user_df[col].apply(fix_value)
    for col in ["pcv", "wc", "rc"]:
        if col in user_df.columns:
            user_df[col] = pd.to_numeric(user_df[col], errors="coerce")

pred = model.predict(user_df)[0]
prob = model.predict_proba(user_df)[0][1]

show_result(pred, prob, disease)

Disease Prediction System
1 - Diabetes
2 - Heart Disease
3 - Kidney Disease


Choose (1/2/3):  1



Loading diabetes data and training model...

Class distribution:
diabetes
0    91500
1     8500
Name: count, dtype: int64



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:26:48] WARNING: /workspace/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:26:48] WARNING: /workspace/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:26:48] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:26:49] WARNING: /workspace/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarnin


DIABETES — Test Results
              precision    recall  f1-score   support

           0       0.98      0.99      0.98     18300
           1       0.88      0.73      0.80      1700

    accuracy                           0.97     20000
   macro avg       0.93      0.86      0.89     20000
weighted avg       0.97      0.97      0.97     20000


✓ Diabetes model is ready.

Enter the patient's information:

